[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Narinder-2006/narinder-flyrank-internship-1/blob/main/work/notebooks/w01_research_question.ipynb)

## 0. Setup (Colab or local)
On Colab this clones the repo (so the starter CSV is available) and moves into it.
Locally it just moves from `work/notebooks/` to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Narinder-2006/narinder-flyrank-internship-1"
REPO_DIR = "narinder-flyrank-internship-1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # move from work/notebooks/ to the repo root

print("Working dir:", os.getcwd())

Working dir: /home/claude/repo


# Week 1 — Research Question

**Intern:** Narinder
**Repo:** narinder-flyrank-internship-1
**Phase:** Setup


## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

Question this lane asks: *which pages should be reviewed first — for refresh, expansion,
protection, pruning, or monitoring?*

I'm picking this over the other three because of what the starter dataset actually looks like
once I load it (see Section 3): a large share of pages are **losing search visibility while
still being visible on page one**. That's a page-level triage problem, not a "what correlates
with clicks in general" problem (Lane 1), not a "what natural groups exist" problem (Lane 3),
and not narrowly a CTR-gap problem (Lane 4) — though CTR context will still show up as one of my
features. Lane 2 is also the lane the reference pipeline (`scripts/01-05`) is already built
around (baseline score → model → ranked queue → Precision@50 evaluation), which means I can
compare my own framing against a working baseline instead of starting from nothing.

I'm treating this as **provisional** — I can confirm or change it by the end of Week 4.

## 2. The question: decision, action, cost of a wrong call

**Search question:** Given a piece of content's search-performance history, how likely is it
that this page is a *high-value refresh candidate* — i.e., a page worth a human's limited
review time this week rather than next quarter?

**Unit of analysis:** one **page** (one row = one `content_id`, a single piece of client
content, aggregated over its own 90-day performance window).

**Output:** a ranked queue of pages, each with a priority score, a suggested action
(refresh / expand / protect / monitor / prune), and a reason code (e.g. "page-one page,
declining 30d trend, high historical demand").

**The decision this improves:** an SEO content team has far more pages than review hours in a
week. Right now that prioritization is manual — someone scans a dashboard and picks pages by
gut feel or recency. This model/analysis replaces that gut-feel scan with a ranked, explainable
list.

**The action someone takes:** a content strategist opens the top N rows of the queue and
schedules those specific pages for a content review this sprint (rewrite, expand, or update)
instead of picking pages arbitrarily.

**Cost of a wrong recommendation:**
- *False positive* (flagged as urgent, but it wasn't really worth it): wastes a writer/strategist's
  hours on a page that wouldn't have moved the needle — the real cost is opportunity cost,
  since that time could have gone to a page that actually mattered.
- *False negative* (a genuinely decaying, high-value page never surfaces near the top): the
  page keeps bleeding impressions/clicks unnoticed until a client or account manager asks why
  traffic dropped — at that point the fix is more expensive and the client relationship takes
  the hit.
- Because both errors cost real time or real traffic, the output has to stay **ranked +
  explained**, not a black-box yes/no. A team can sanity-check "page-one page, down 40% over
  30 days" in a way they can't sanity-check a bare probability.

**Why data/ML helps here at all:** with hundreds to thousands of pages per client and 30+
signals per page (position, CTR, impressions/clicks trend, freshness, word count, intent...),
no one is manually recomputing a 30-day-vs-prior-30-day decline rate across every single page
every week. A transparent baseline rule can already do a first pass (that's `scripts/02`); the
open question for my capstone is whether a learned model, trained on which pages *actually* got
reviewed/recovered historically, can rank better than that hand-written rule — the same
Precision@50 comparison the reference pipeline already sets up (baseline ≈0.24 → model
0.68–0.74 on this dataset).

## 3. Quick look at the data (real numbers)

Loading the bundled starter dataset (`data/raw/content_refresh_anonymized.csv`) to back the
lane choice with numbers instead of a hunch.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("rows (pages):", len(df))
print("distinct clients:", df["client_id"].nunique())
df.head(3)

rows (pages): 30000
distinct clients: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


**Number 1 — scale.** The starter sample has 30,000 pages across 32 clients — small enough
to run entirely on a laptop this week, but large enough that a per-client manual review is
clearly not happening today. That gap is exactly what a ranked-queue tool would fill.

In [3]:
trend_counts = df["trend_direction"].value_counts()
trend_share = (trend_counts / len(df) * 100).round(1)
print(trend_counts)
print()
print(trend_share)

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: count, dtype: float64


**Number 2 — how much is actually declining.** 16,262 of 30,000 pages (about 54%) carry a
`down` trend label over the last-30-vs-prior-30-day window. Just knowing "over half the
inventory is declining" isn't itself a triage list — the real question is *which* of those
declining pages are worth touching first, which is what the ranking/scoring step is for.

In [4]:
page_one_decline = df[(df["avg_position"] <= 10) & (df["trend_direction"] == "down")]
share = round(100 * len(page_one_decline) / len(df), 1)
print(f"page-one pages that are also declining: {len(page_one_decline)} rows, {share}% of all pages")

# what the same pages look like on the CTR side, by position tier
df.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)

page-one pages that are also declining: 7319 rows, 24.4% of all pages


position_tier
top_3       1.483611
page_1      0.652467
striking    0.323239
page_3_5    0.222484
deep        0.150212
Name: ctr, dtype: float64

**Number 3 — the highest-value slice.** 7,319 pages (about 24%) are both **visible on page
one** (avg position ≤ 10) *and* **trending down** — this is literally the "page-one decay risk"
baseline idea the lane guide names as a starting signal for Lane 2. These are the pages where a
small ranking mistake is most expensive: they already earn real clicks (`top_3`/`page_1`
position tiers show CTR well above the `deep` tier), so losing rank here costs more traffic
than a page buried on page 3. That's a concrete, decision-relevant slice worth building the
priority score around — not just "most pages are declining," but "here is the subset with real
traffic at stake if we get the priority wrong.

## 4. Careful words: what I can and can't claim

**What I can claim, once I've built and validated something:**
- observed, *associational* patterns between a page's own historical signals (position trend,
  CTR, freshness, word count, impression volume) and whether it later showed a recovery/decline;
- a ranked, explainable priority list that beats a transparent hand-written baseline on a
  held-out set, by a stated metric (Precision@K);
- reason codes tied to specific, auditable signals on that page (not an opaque score).

**What I cannot claim:**
- that I've identified a Google ranking-algorithm factor — I only have my own client's
  observed signals, not Google's internals;
- that refreshing a flagged page will *cause* a recovery — this is an observational dataset,
  not an experiment, so I can say a page matches a historical pattern associated with
  recovery, not that action X will produce outcome Y;
- that the model is right about any single page — it's a prioritization aid for a human
  reviewer, not an auto-pilot decision-maker;
- anything client-identifying — all analysis and any public write-up stays in
  observed/measured/directional/decision-support language, per `DATA_USE.md`.

I also need to watch for the two failure modes the lane guide calls out directly for Lane 2:
treating "declining" as guaranteed refresh ROI, and accidentally using future-window metrics
(the "after" side of a trend) as a model feature, which would leak the label into the inputs.

## 5. Self-check

- [x] Picked one of the four predefined lanes (Lane 2 — Refresh / Content Opportunity Scoring),
      not freestyle — and stated why, including why the other three fit less well.
- [x] Named the decision this improves (weekly content-review prioritization for an SEO team)
      and the action someone takes from it (schedule the top-N flagged pages for review).
- [x] Named the cost of a wrong call in both directions (wasted review time vs. a real page
      quietly bleeding traffic until a client notices).
- [x] Backed the lane choice with real numbers from the starter dataset, loaded live in a code
      cell: 30,000 rows / 32 clients, ~54% of pages trending down, ~24% both page-one visible
      *and* declining.
- [x] Explained why this isn't just "train a model" — the reference pipeline already trains a
      model; the actual work is problem framing, a defensible baseline, leakage-safe features,
      and a validation design that maps to the real decision (Precision@K on a held-out,
      client-disjoint split).
- [x] Used careful, non-causal, non-client-identifying language throughout (Section 4).
- [ ] Still open for Week 2+: which exact features go into the baseline score, and whether a
      logistic regression / decision tree / random forest beats it — that's next week's work,
      not this notebook's job.

**Lane choice status:** provisional — open to revisiting through the end of Week 4, per the
assignment brief.